In [12]:
import numpy as np
from pathlib import Path
from numpy.linalg import norm
from sklearn.metrics import roc_curve
from tqdm import tqdm


In [13]:
#  x-vectors
XVEC_PATH = Path("../outputs/xvectors/xvectors.npy")
UTT_PATH  = Path("../outputs/xvectors/utterances.npy")

# VoxCeleb1 test protocol
PROTOCOL_PATH = Path("../data/voxceleb1/test/veri_test2.txt")


In [14]:
X = np.load(XVEC_PATH)
utterances = np.load(UTT_PATH, allow_pickle=True)

print("X shape:", X.shape)
print("Utterances:", len(utterances))
print("Example:", utterances[0])


X shape: (4874, 512)
Utterances: 4874
Example: id10295/nt7dNRvlEHE/00005.wav


In [15]:
X_norm = X / norm(X, axis=1, keepdims=True)


In [16]:
utt2idx = {utt: i for i, utt in enumerate(utterances)}

print("Mapping example:")
print(utterances[0], "->", utt2idx[utterances[0]])


Mapping example:
id10295/nt7dNRvlEHE/00005.wav -> 0


In [17]:
trials = []
labels = []

with open(PROTOCOL_PATH) as f:
    for line in f:
        lab, u1, u2 = line.strip().split()
        trials.append((u1, u2))
        labels.append(int(lab))

labels = np.array(labels)

print("Total trials:", len(trials))
print("Positive (same speaker):", np.sum(labels == 1))
print("Negative (diff speaker):", np.sum(labels == 0))


Total trials: 37611
Positive (same speaker): 18802
Negative (diff speaker): 18809


In [18]:
scores = np.zeros(len(trials), dtype=np.float32)

for k, (utt1, utt2) in enumerate(tqdm(trials)):
    i = utt2idx[utt1]
    j = utt2idx[utt2]
    scores[k] = np.dot(X_norm[i], X_norm[j])


100%|██████████| 37611/37611 [00:00<00:00, 892060.44it/s]


In [19]:
# Pick a random utterance
u = utterances[0]
spk = u.split("/")[0]

same_spk = [x for x in utterances if x.startswith(spk)]
diff_spk = [x for x in utterances if not x.startswith(spk)]

i = utt2idx[same_spk[0]]
j1 = utt2idx[same_spk[1]]
j2 = utt2idx[diff_spk[0]]

print("Same speaker cosine:", np.dot(X_norm[i], X_norm[j1]))
print("Different speaker cosine:", np.dot(X_norm[i], X_norm[j2]))


Same speaker cosine: 0.94930315
Different speaker cosine: 0.9223137


In [20]:
import pickle
import numpy as np

# save protocol components in outputs/protocol/protocol.pkl
OUT_DIR = Path("../outputs/protocol")
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUT_DIR / "protocol.pkl", "wb") as f:
    pickle.dump({
        "utterances": utterances,
        "utt2idx": utt2idx,
        "trials": trials
    }, f)
np.save(OUT_DIR / "labels.npy", labels)

In [22]:
p_target=0.01
c_miss=1.0
c_fa=1.0
fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
fnr = 1 - tpr
dcf = c_miss * fnr * p_target + c_fa * fpr * (1 - p_target)
min_dcf = np.min(dcf)
eer = fpr[np.nanargmin(np.abs(fnr - fpr))]
print(f"TDNN_Xvector-Cosine - EER: {eer * 100:.2f}%")
print(f"TDNN_Xvector-Cosine-  MinDCF : {min_dcf:.4f}")

TDNN_Xvector-Cosine - EER: 8.88%
TDNN_Xvector-Cosine-  MinDCF : 0.0065
